# ODGO setup — On-premises Data Gateway Observability

This notebook installs or upgrades ODGO in **the workspace where it runs**:

| Item | Name |
|---|---|
| Lakehouse (with schemas) | `lh_gateway_monitor` |
| Notebooks | `nb_gwmon_lib`, `nb_gwmon_ingest` (scheduled every 6 hours), `nb_gwmon_maintenance` (scheduled once a day) |
| Semantic model (Direct Lake) and report | `Gateway Monitor` |

**Before you run it:** the workspace must be assigned to a Fabric capacity, and you need the Admin or Member role.

**To run it:** optionally fill in the parameters of the next cell, then select **Run all**. The setup takes a few
minutes. The last cell prints the command that installs the agent on your gateway servers.

**To upgrade ODGO:** run this notebook again. It downloads the version set in `source` and updates the items in
place. Data is kept.

Documentation: https://github.com/Pulsweb/ODGO

In [ ]:
# Agent access: the identity of the gateway agents needs the Contributor role on this workspace. Either leave
# agent_principal_id empty and add the identity yourself (workspace > Manage access > Add people or groups > type
# the name of the app registration or managed identity > Contributor), or set it to the Object ID of the identity:
# Microsoft Entra admin center > Enterprise apps > your app > Overview > Object ID. Not the Application (client) ID,
# and not the Object ID shown under App registrations.
agent_principal_id = ""
agent_principal_type = "ServicePrincipal"  # or "Group" for the Object ID of a security group of agent identities
# Optional: Application (client) ID of the app registration, used to complete the printed install command.
agent_client_id = ""
# ODGO version to install: a .zip of the repository (branch or release archive).
source = "https://github.com/Pulsweb/ODGO/archive/refs/heads/main.zip"
# How often nb_gwmon_ingest runs, in minutes (nb_gwmon_maintenance runs once a day, between two of its runs).
# Above 360 minutes, also raise gold.lateAfterMinutes in processing.json (see docs/configuration.md).
ingest_interval_minutes = 360
# Run the ingestion once now (creates the tables) and frame the semantic model.
run_first_ingestion = True

In [ ]:
import base64
import io
import json
import os
import re
import time
import urllib.request
import zipfile
from datetime import datetime, timedelta, timezone
from typing import Any, Callable, Dict, List, Optional

SETUP_VERSION = "1.0.0"
DEFAULT_SOURCE = "https://github.com/Pulsweb/ODGO/archive/refs/heads/main.zip"
LAKEHOUSE_NAME = "lh_gateway_monitor"
NOTEBOOKS = ("nb_gwmon_lib", "nb_gwmon_ingest", "nb_gwmon_maintenance")
MODEL_NAME = "Gateway Monitor"
REPORT_NAME = "Gateway Monitor"
MODEL_FOLDER = "powerbi/GatewayMonitor.SemanticModel"
REPORT_FOLDER = "powerbi/GatewayMonitor.Report"
SETUP_NOTEBOOK = "fabric/ODGO_Setup.ipynb"
SOURCE_PREFIXES = ("fabric/notebooks/", MODEL_FOLDER + "/", REPORT_FOLDER + "/", SETUP_NOTEBOOK)


class SetupError(RuntimeError):
    """A setup step failed; the message says what to check."""


def load_source(source: str) -> Dict[str, bytes]:
    """Files used by the setup, keyed by repository path, from a .zip (URL or file path) or a repository folder."""
    files: Dict[str, bytes] = {}
    if os.path.isdir(source):
        for root, _, names in os.walk(source):
            for name in names:
                full = os.path.join(root, name)
                with open(full, "rb") as handle:
                    files[os.path.relpath(full, source).replace(os.sep, "/")] = handle.read()
    else:
        if re.match(r"^https?://", source, re.IGNORECASE):
            with urllib.request.urlopen(source, timeout=300) as response:
                data = response.read()
        else:
            with open(source, "rb") as handle:
                data = handle.read()
        with zipfile.ZipFile(io.BytesIO(data)) as archive:
            names = [name for name in archive.namelist() if not name.endswith("/")]
            tops = {name.split("/", 1)[0] for name in names}
            prefix = tops.pop() + "/" if len(tops) == 1 and all("/" in name for name in names) else ""
            for name in names:
                files[name[len(prefix):]] = archive.read(name)
    selected = {path: content for path, content in files.items() if path.startswith(SOURCE_PREFIXES)}
    required = [f"fabric/notebooks/{name}.Notebook/notebook-content.py" for name in NOTEBOOKS]
    required += [f"{MODEL_FOLDER}/definition.pbism", f"{REPORT_FOLDER}/definition.pbir"]
    missing = [path for path in required if path not in selected]
    if missing:
        raise SetupError(f"{source} doesn't look like an ODGO repository or archive (missing {', '.join(missing)}).")
    return selected


class FabricApi:
    """Small Fabric / Power BI REST helper: errors, throttling, paging and long-running operations."""

    def __init__(self, client, sleep: Callable[[float], None] = time.sleep, log: Callable[[str], None] = print):
        self.client, self.sleep, self.log = client, sleep, log

    def request(self, method: str, path: str, body: Any = None, allowed=()):
        for attempt in range(8):
            kwargs = {"json": body} if body is not None else {}
            response = self.client.request(method, path, **kwargs)
            status = response.status_code
            if status == 429 and attempt < 7:
                self.sleep(min(int(response.headers.get("Retry-After") or 10), 60))
                continue
            if status >= 400 and status not in allowed:
                raise SetupError(f"{method} {path} failed with HTTP {status}: {response.text[:1000]}")
            return response
        raise SetupError(f"{method} {path} is still throttled after several retries")

    @staticmethod
    def parse(response) -> Any:
        return response.json() if response.content else None

    def call(self, method: str, path: str, body: Any = None, timeout_minutes: int = 30) -> Any:
        """Request; waits for long-running operations (202 + Location) and returns the parsed result."""
        response = self.request(method, path, body)
        if response.status_code != 202 or not response.headers.get("Location"):
            return self.parse(response)
        location = response.headers["Location"]
        delay = int(response.headers.get("Retry-After") or 3)
        for _ in range(max(1, int(timeout_minutes * 60 / max(delay, 1)))):
            self.sleep(delay)
            state_response = self.request("GET", location)
            state = self.parse(state_response) or {}
            status = str(state.get("status", "")).lower()
            if status == "succeeded":
                result_location = state_response.headers.get("Location")
                return self.parse(self.request("GET", result_location)) if result_location else None
            if status in ("failed", "cancelled"):
                raise SetupError(f"{method} {path} failed: {json.dumps(state.get('error'))}")
        raise SetupError(f"{method} {path} didn't complete within {timeout_minutes} minutes")

    def list(self, path: str) -> List[Dict[str, Any]]:
        items, next_path = [], path
        while next_path:
            page = self.parse(self.request("GET", next_path)) or {}
            items.extend(page.get("value") or [])
            next_path = page.get("continuationUri")
        return items


def part(path: str, data: bytes) -> Dict[str, str]:
    return {"path": path, "payload": base64.b64encode(data).decode("ascii"), "payloadType": "InlineBase64"}


def find_item(api: FabricApi, workspace_id: str, name: str, item_type: str) -> Optional[Dict[str, Any]]:
    items = api.list(f"v1/workspaces/{workspace_id}/items?type={item_type}")
    return next((item for item in items if item.get("displayName") == name), None)


def upsert_item(api: FabricApi, workspace_id: str, name: str, item_type: str, definition: Dict[str, Any],
                description: str = "") -> str:
    """Creates the item with its definition, or updates the definition of the existing item (matched by name)."""
    item = find_item(api, workspace_id, name, item_type)
    if item is None:
        body = {"displayName": name, "type": item_type, "description": description, "definition": definition}
        api.call("POST", f"v1/workspaces/{workspace_id}/items", body)
        item = find_item(api, workspace_id, name, item_type)
        if item is None:
            raise SetupError(f"{item_type} '{name}' wasn't created")
        api.log(f"   created {item_type} '{name}'")
    else:
        api.call("POST", f"v1/workspaces/{workspace_id}/items/{item['id']}/updateDefinition", {"definition": definition})
        api.log(f"   updated {item_type} '{name}'")
    return item["id"]


def ensure_lakehouse(api: FabricApi, workspace_id: str, name: str) -> str:
    item = find_item(api, workspace_id, name, "Lakehouse")
    if item is None:
        api.call("POST", f"v1/workspaces/{workspace_id}/lakehouses", {
            "displayName": name, "description": "ODGO data: landing files and bronze, silver, gold, ops tables",
            "creationPayload": {"enableSchemas": True}})
        for _ in range(10):
            item = find_item(api, workspace_id, name, "Lakehouse")
            if item:
                break
            api.sleep(3)
        if item is None:
            raise SetupError(f"Lakehouse '{name}' wasn't created")
        api.log(f"   created lakehouse '{name}'")
    else:
        api.log(f"   lakehouse '{name}' already exists")
    return item["id"]


META_BLOCK = re.compile(r"# METADATA \*+\n\n((?:# META.*\n)+)")


def bind_notebook(content: str, workspace_id: str, lakehouse_id: str, lakehouse_name: str) -> str:
    """Sets the default lakehouse in the notebook-level metadata of a Fabric Git-format notebook."""
    text = content.replace("\r\n", "\n")
    match = META_BLOCK.search(text)
    if match is None:
        raise SetupError("notebook source without a metadata block")
    meta = json.loads("\n".join(re.sub(r"^# META ?", "", line) for line in match.group(1).splitlines()))
    meta["dependencies"] = {"lakehouse": {
        "default_lakehouse": lakehouse_id, "default_lakehouse_name": lakehouse_name,
        "default_lakehouse_workspace_id": workspace_id, "known_lakehouses": [{"id": lakehouse_id}]}}
    block = "\n".join("# META " + line for line in json.dumps(meta, indent=2).splitlines()) + "\n"
    return text[:match.start(1)] + block + text[match.end(1):]


def notebook_definition(files: Dict[str, bytes], name: str, workspace_id: str, lakehouse_id: str) -> Dict[str, Any]:
    content = files[f"fabric/notebooks/{name}.Notebook/notebook-content.py"].decode("utf-8")
    bound = bind_notebook(content, workspace_id, lakehouse_id, LAKEHOUSE_NAME)
    return {"format": "fabricGitSource", "parts": [part("notebook-content.py", bound.encode("utf-8"))]}


def notebook_description(files: Dict[str, bytes], name: str) -> str:
    platform = files.get(f"fabric/notebooks/{name}.Notebook/.platform")
    return json.loads(platform.decode("utf-8")).get("metadata", {}).get("description", "") if platform else ""


def folder_parts(files: Dict[str, bytes], folder: str, transform=None) -> List[Dict[str, str]]:
    parts = []
    for path in sorted(p for p in files if p.startswith(folder + "/")):
        relative = path[len(folder) + 1:]
        if relative == ".platform" or relative.startswith(".pbi/"):
            continue
        data = transform(relative, files[path]) if transform else files[path]
        parts.append(part(relative, data))
    return parts


def model_definition(files: Dict[str, bytes], workspace_id: str, lakehouse_id: str) -> Dict[str, Any]:
    tokens = {"{{FABRIC_WORKSPACE_ID}}": workspace_id, "{{FABRIC_LAKEHOUSE_ID}}": lakehouse_id}

    def transform(relative: str, data: bytes) -> bytes:
        if not relative.endswith(".tmdl"):
            return data
        text = data.decode("utf-8")
        for token, value in tokens.items():
            text = text.replace(token, value)
        left = sorted(set(re.findall(r"\{\{[A-Z0-9_]+\}\}", text)))
        if left:
            raise SetupError(f"unresolved token(s) {', '.join(left)} in {relative}")
        return text.encode("utf-8")

    return {"parts": folder_parts(files, MODEL_FOLDER, transform)}


def report_definition(files: Dict[str, bytes], model_id: str) -> Dict[str, Any]:
    def transform(relative: str, data: bytes) -> bytes:
        if relative != "definition.pbir":
            return data
        document = json.loads(data.decode("utf-8-sig"))
        document["datasetReference"] = {"byConnection": {"connectionString": f"semanticmodelid={model_id}"}}
        return json.dumps(document, indent=2).encode("utf-8")

    return {"parts": folder_parts(files, REPORT_FOLDER, transform)}


PRINCIPAL_ID_HINT = ("agent_principal_id must be the Object ID of the agent identity: for an app registration, the "
                     "Object ID shown under Enterprise apps, not its Application (client) ID nor the Object ID shown "
                     "under App registrations. Or leave agent_principal_id empty and add the identity yourself: "
                     "workspace > Manage access > Add people or groups > Contributor.")


def ensure_contributor(api: FabricApi, workspace_id: str, principal_id: str, principal_type: str) -> str:
    """Grants Contributor to the agent identity (keeps a higher role if it already has one)."""
    if not re.fullmatch(r"[0-9a-fA-F]{8}-(?:[0-9a-fA-F]{4}-){3}[0-9a-fA-F]{12}", principal_id):
        raise SetupError(f"'{principal_id}' isn't an Object ID. {PRINCIPAL_ID_HINT}")
    assignments = api.list(f"v1/workspaces/{workspace_id}/roleAssignments")
    current = next((a for a in assignments
                    if str((a.get("principal") or {}).get("id", "")).lower() == principal_id.lower()), None)
    if current and current.get("role") in ("Admin", "Member", "Contributor"):
        return current["role"]
    if current:
        api.call("PATCH", f"v1/workspaces/{workspace_id}/roleAssignments/{current['id']}", {"role": "Contributor"})
    else:
        try:
            api.call("POST", f"v1/workspaces/{workspace_id}/roleAssignments",
                     {"principal": {"id": principal_id, "type": principal_type}, "role": "Contributor"})
        except SetupError as exc:
            raise SetupError(f"couldn't give the Contributor role to {principal_type} {principal_id} ({exc}). "
                             f"{PRINCIPAL_ID_HINT}") from exc
    return "Contributor"


def run_notebook(api: FabricApi, workspace_id: str, notebook_id: str, timeout_minutes: int = 60,
                 poll_seconds: int = 15) -> Dict[str, Any]:
    response = api.request("POST", f"v1/workspaces/{workspace_id}/items/{notebook_id}/jobs/instances?jobType=RunNotebook")
    location = response.headers.get("Location")
    if not location:
        raise SetupError("the notebook run didn't return a job location")
    for _ in range(int(timeout_minutes * 60 / poll_seconds) + 1):
        api.sleep(poll_seconds)
        job = api.parse(api.request("GET", location)) or {}
        status = job.get("status")
        if status in ("Completed", "Failed", "Cancelled", "Deduped"):
            if status != "Completed":
                raise SetupError(f"nb_gwmon_ingest ended with status {status}: {json.dumps(job.get('failureReason'))}. "
                                 "Open the run in the Monitor hub for details, fix the cause and run the setup again.")
            return job
    raise SetupError(f"nb_gwmon_ingest didn't complete within {timeout_minutes} minutes")


def refresh_model(api: FabricApi, workspace_id: str, model_id: str, timeout_minutes: int = 30,
                  poll_seconds: int = 10) -> Dict[str, Any]:
    """Frames the Direct Lake model (full refresh) and waits for the result."""
    base = f"v1.0/myorg/groups/{workspace_id}/datasets/{model_id}/refreshes"
    response = api.request("POST", base, {"type": "full"})
    location = response.headers.get("Location")
    for _ in range(int(timeout_minutes * 60 / poll_seconds) + 1):
        api.sleep(poll_seconds)
        if location:
            state = api.parse(api.request("GET", location)) or {}
        else:
            state = ((api.parse(api.request("GET", f"{base}?$top=1")) or {}).get("value") or [{}])[0]
        status = state.get("status")
        if status == "Completed":
            return state
        if status in ("Failed", "Cancelled", "Disabled", "TimedOut"):
            raise SetupError(f"semantic model refresh {status}: {state.get('serviceExceptionJson') or state.get('messages')}")
    raise SetupError(f"the semantic model refresh didn't complete within {timeout_minutes} minutes")


def ensure_schedule(api: FabricApi, workspace_id: str, item_id: str, configuration: Dict[str, Any]) -> str:
    """Creates or updates the first schedule of a notebook (job type RunNotebook, DefaultJob as fallback)."""
    body = {"enabled": True, "configuration": configuration}
    errors = []
    for job_type in ("RunNotebook", "DefaultJob"):
        base = f"v1/workspaces/{workspace_id}/items/{item_id}/jobs/{job_type}/schedules"
        try:
            existing = api.list(base)
            if existing:
                api.call("PATCH", f"{base}/{existing[0]['id']}", body)
            else:
                api.call("POST", base, body)
            return job_type
        except SetupError as exc:
            errors.append(str(exc))
    raise SetupError("couldn't create the schedule: " + " | ".join(errors))


def schedule_start(now: datetime, interval_minutes: int) -> datetime:
    """Next ingestion run: runs stay on a grid of interval_minutes that starts at 00:00 UTC.

    The start must be in the future: Fabric runs a schedule immediately when its start time is in the past.
    """
    now = now.astimezone(timezone.utc).replace(tzinfo=None)
    midnight = now.replace(hour=0, minute=0, second=0, microsecond=0)
    elapsed = (now - midnight).total_seconds() / 60
    return midnight + timedelta(minutes=interval_minutes * (int(elapsed // interval_minutes) + 1))


def maintenance_time(interval_minutes: int) -> str:
    """Daily maintenance slot (UTC, hh:mm): halfway between two ingestion runs of the grid."""
    minutes = (min(interval_minutes, 1440) // 2) % 1440
    return f"{minutes // 60:02d}:{minutes % 60:02d}"


def describe_interval(minutes: int) -> str:
    if minutes % 60:
        return f"{minutes} minutes"
    return "hour" if minutes == 60 else f"{minutes // 60} hours"


def install_command(tenant_id: Optional[str], client_id: Optional[str], workspace_id: str, lakehouse_id: str) -> str:
    return (f"& $installer -WorkspaceId {workspace_id} -LakehouseId {lakehouse_id} "
            f"-TenantId {tenant_id or '<tenant-id>'} -ClientId {client_id or '<client-id>'}")


def agent_download_commands(zip_url: str) -> List[str]:
    """PowerShell lines that download the agent of the same ODGO version and locate its installer."""
    return [f"Invoke-WebRequest '{zip_url}' -OutFile odgo.zip",
            "Remove-Item odgo -Recurse -Force -ErrorAction Ignore; Expand-Archive odgo.zip odgo",
            "$installer = (Get-ChildItem odgo -Recurse -Filter Install-Agent.ps1 | Select-Object -First 1).FullName"]


def setup_version_of(files: Dict[str, bytes]) -> Optional[str]:
    match = re.search(rb'SETUP_VERSION = \\?"([0-9.]+)\\?"', files.get(SETUP_NOTEBOOK, b""))
    return match.group(1).decode("ascii") if match else None


def deploy(api: FabricApi, workspace_id: str, files: Dict[str, bytes], *, agent_principal_id: str = "",
           agent_principal_type: str = "ServicePrincipal", agent_client_id: str = "", tenant_id: Optional[str] = None,
           ingest_interval_minutes: int = 360, run_first_ingestion: bool = True,
           agent_zip_url: str = DEFAULT_SOURCE, today: Optional[datetime] = None) -> Dict[str, Any]:
    log = api.log
    warnings: List[str] = []
    interval = int(ingest_interval_minutes)
    if interval < 1:
        raise SetupError("ingest_interval_minutes must be a positive number of minutes (360 = every 6 hours).")
    downloaded_setup = setup_version_of(files)
    if downloaded_setup and downloaded_setup != SETUP_VERSION:
        warnings.append(f"This setup notebook is version {SETUP_VERSION} but the downloaded ODGO version ships setup "
                        f"{downloaded_setup}: import the newer fabric/ODGO_Setup.ipynb and run it.")

    workspace = api.call("GET", f"v1/workspaces/{workspace_id}") or {}
    if not workspace.get("capacityId"):
        raise SetupError("This workspace isn't assigned to a Fabric capacity. Assign one (Workspace settings > "
                         "License info) and run the setup again.")
    log(f"Workspace '{workspace.get('displayName')}' ({workspace_id})")

    log("1/7 Lakehouse")
    lakehouse_id = ensure_lakehouse(api, workspace_id, LAKEHOUSE_NAME)

    log("2/7 Notebooks")
    notebook_ids = {name: upsert_item(api, workspace_id, name, "Notebook",
                                      notebook_definition(files, name, workspace_id, lakehouse_id),
                                      notebook_description(files, name))
                    for name in NOTEBOOKS}

    log("3/7 Agent permission")
    agent_role = None
    if agent_principal_id.strip():
        agent_role = ensure_contributor(api, workspace_id, agent_principal_id.strip(), agent_principal_type)
        log(f"   {agent_principal_type} {agent_principal_id.strip()} has the {agent_role} role")
    else:
        log("   skipped: add the agent identity as Contributor in Manage access (or set agent_principal_id)")

    log("4/7 First ingestion (creates the tables)")
    if run_first_ingestion:
        run_notebook(api, workspace_id, notebook_ids["nb_gwmon_ingest"])
        log("   nb_gwmon_ingest completed")
    else:
        log("   skipped")

    log("5/7 Semantic model")
    model_id = upsert_item(api, workspace_id, MODEL_NAME, "SemanticModel",
                           model_definition(files, workspace_id, lakehouse_id), "ODGO model (Direct Lake on OneLake)")
    if run_first_ingestion:
        try:
            refresh_model(api, workspace_id, model_id)
            log("   framed")
        except SetupError as exc:
            warnings.append(f"Semantic model framing failed ({exc}). It's retried automatically when the Gold tables "
                            "change; check the model's refresh history.")

    log("6/7 Report")
    report_id = upsert_item(api, workspace_id, REPORT_NAME, "Report", report_definition(files, model_id),
                            "ODGO report")

    log("7/7 Schedules")
    first_run = schedule_start(today or datetime.now(timezone.utc), interval)
    start = first_run.strftime("%Y-%m-%dT%H:%M:%S")
    end = (first_run + timedelta(days=3652)).strftime("%Y-%m-%dT%H:%M:%S")
    ensure_schedule(api, workspace_id, notebook_ids["nb_gwmon_ingest"], {
        "type": "Cron", "interval": interval, "startDateTime": start, "endDateTime": end, "localTimeZoneId": "UTC"})
    ensure_schedule(api, workspace_id, notebook_ids["nb_gwmon_maintenance"], {
        "type": "Daily", "times": [maintenance_time(interval)], "startDateTime": start, "endDateTime": end,
        "localTimeZoneId": "UTC"})
    log(f"   nb_gwmon_ingest every {describe_interval(interval)}, nb_gwmon_maintenance once a day")

    return {
        "workspace_id": workspace_id, "lakehouse_id": lakehouse_id, "notebook_ids": notebook_ids,
        "semantic_model_id": model_id, "report_id": report_id, "agent_role": agent_role, "warnings": warnings,
        "agent_download": agent_download_commands(agent_zip_url),
        "install_command": install_command(tenant_id, agent_client_id, workspace_id, lakehouse_id),
        "install_command_managed_identity": (f"& $installer -WorkspaceId {workspace_id} "
                                             f"-LakehouseId {lakehouse_id} -ManagedIdentity"),
    }


def current_tenant_id() -> Optional[str]:
    """Tenant ID read from the notebook user's token (used only to pre-fill the install command)."""
    try:
        import notebookutils  # type: ignore
        payload = notebookutils.credentials.getToken("pbi").split(".")[1]
        payload += "=" * (-len(payload) % 4)
        return json.loads(base64.urlsafe_b64decode(payload)).get("tid")
    except Exception:
        return None


def current_workspace_id() -> str:
    try:
        import sempy.fabric as fabric  # type: ignore
        return fabric.get_workspace_id()
    except Exception:
        import notebookutils  # type: ignore
        return notebookutils.runtime.context["currentWorkspaceId"]


def print_summary(result: Dict[str, Any]) -> None:
    print("\nODGO is installed.\n")
    for warning in result["warnings"]:
        print(f"WARNING: {warning}\n")
    step = "Next"
    if not result.get("agent_role"):
        print("Next: give the identity of the gateway agents the Contributor role on this workspace: Manage access >")
        print("Add people or groups > type the name of the app registration (or managed identity) > Contributor.\n")
        step = "Then"
    print(f"{step}: install the agent on each gateway server. Paste these lines in an elevated PowerShell 7 session")
    print("(you'll be prompted for the client secret value of the app registration):\n")
    for line in result["agent_download"] + [result["install_command"]]:
        print(f"    {line}")
    print("\nGateway on an Azure VM or an Azure Arc-enabled server (managed identity, no secret): replace the last line with\n")
    print(f"    {result['install_command_managed_identity']}\n")
    print("Then open the 'Gateway Monitor' report: the Ingestion Health page shows each server once nb_gwmon_ingest has")
    print("processed its first upload. To see it before the next scheduled run, run nb_gwmon_ingest yourself.")

In [ ]:
import sempy.fabric as fabric  # Semantic Link, preinstalled in Fabric

print(f"ODGO setup {SETUP_VERSION}: downloading {source}")
setup_result = deploy(
    FabricApi(fabric.FabricRestClient()), current_workspace_id(), load_source(source),
    agent_principal_id=agent_principal_id, agent_principal_type=agent_principal_type, agent_client_id=agent_client_id,
    tenant_id=current_tenant_id(), ingest_interval_minutes=ingest_interval_minutes, run_first_ingestion=run_first_ingestion,
    agent_zip_url=source if source.lower().startswith("https://") else DEFAULT_SOURCE)
print_summary(setup_result)